# NOCTIS — Image generation and LoRA training

**Course project:** *Postcards from My Jungle*. This notebook fine-tunes an existing Stable Diffusion 1.5 **diffusion model** with LoRA on real wildlife portraits from [HugGAN/AFHQ](https://huggingface.co/datasets/huggan/AFHQ), then generates **50 fictional jungle-animal images** and exports them with the exact filenames that the webpage expects.

### Before running
1. In Google Colab, choose **Runtime → Change runtime type → T4 GPU** (or another NVIDIA GPU).
2. Choose **Runtime → Run all**. The model and dataset must download over the internet, so online execution is required.
3. After images are generated, get `images-generated.zip`, extract it, and **replace every matching image** in your site's `images/` folder. Set `DEMO_MODE = false` in `script.js` **only after both** image and sound files have been replaced.

**Important:** LoRA trains adapter weights rather than a diffusion model from scratch. It is real fine-tuning of the diffusion model, and requires a GPU. It is not the same as generating images using a text prompt alone. This notebook has not been executed in this package; review its outputs in Colab. Training can take a substantial amount of GPU time.

In [ ]:
import torch, os
assert torch.cuda.is_available(), "A GPU is required. In Colab: Runtime → Change runtime type → T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

## 1. Install dependencies

Pin matching training and inference libraries for reproducibility. If you encounter package-import errors immediately after installing, use **Runtime → Restart session**, then resume below the installation cell.

In [ ]:
%pip -q install "diffusers==0.35.1" "transformers==4.49.0" "accelerate==1.5.2" "peft==0.15.2" "datasets==3.5.1" "huggingface_hub>=0.30,<1" "safetensors>=0.4" "ftfy" "scipy" "sentencepiece" "tensorboard"

## 2. Collect a real animal dataset

We stream the AFHQ dataset (instead of downloading every image) and select photos with the **wild** label. Change `DATASET_SIZE` to **80**, **160**, or **320** to compare results. The notebook creates a properly captioned local `imagefolder` dataset for the LoRA training script.

The training images are source photographs; **do not include them in the GitHub repository** unless their license allows redistribution. Only the model-generated images belong in `images/`.

In [ ]:
from datasets import load_dataset
from pathlib import Path
from PIL import Image
import json, random

DATASET_SIZE = 160  # Try 80, 160, 320. Larger datasets take longer.
RESOLUTION = 384  # 384 works better than 512 on limited T4 GPU memory.
MAX_TRAIN_STEPS = 500  # Try 250, 500, 800 for a simple comparison.
SEED = 42
TRAIN_DIR = Path('/content/noctis-training')
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

stream = load_dataset('huggan/AFHQ', split='train', streaming=True).shuffle(seed=SEED, buffer_size=1000)
label_feature = stream.features['label']
wild_id = label_feature.str2int('wild')
records=[]
for example in stream:
    if int(example['label']) != wild_id:
        continue
    filename=f'wild-{len(records):04d}.jpg'
    image=example['image'].convert('RGB')
    image.thumbnail((RESOLUTION, RESOLUTION))
    # square padded portrait for training without deforming the source
    canvas=Image.new('RGB',(RESOLUTION,RESOLUTION),(37,49,37))
    canvas.paste(image,((RESOLUTION-image.width)//2,(RESOLUTION-image.height)//2))
    canvas.save(TRAIN_DIR/filename,quality=91)
    records.append({'file_name':filename, 'text':'portrait photograph of an unusual wild jungle animal, lush forest, natural texture, noctisfauna'})
    if len(records)>=DATASET_SIZE:
        break
assert len(records)==DATASET_SIZE, f'Found only {len(records)} suitable images.'
with (TRAIN_DIR/'metadata.jsonl').open('w',encoding='utf-8') as fh:
    for record in records: fh.write(json.dumps(record)+'\n')
print('Dataset images:',len(records),' | folder:',TRAIN_DIR)
print('First:',records[0])

In [ ]:
from IPython.display import display
files=sorted(TRAIN_DIR.glob('*.jpg'))
for f in files[:6]:
    display(Image.open(f).resize((160,160)))

## 3. Fine-tune Stable Diffusion with LoRA

Uses Hugging Face's official `train_text_to_image_lora.py` from the matching **v0.35.1** Diffusers tag. LoRA trains a small adapter on the wildlife photos. On low-memory hardware, keep 384 pixels, batch size 1, gradient checkpointing, and mixed precision. If the base model asks for a license confirmation, visit the model page and sign into Hugging Face in Colab via `huggingface_hub.login()`.

**Experiment notes to record:** chosen dataset size, resolution, total steps, a screenshot of your training log, and a sample generated image from each configuration.

In [ ]:
from urllib.request import urlretrieve
from pathlib import Path
SCRIPT=Path('/content/train_text_to_image_lora.py')
url='https://raw.githubusercontent.com/huggingface/diffusers/v0.35.1/examples/text_to_image/train_text_to_image_lora.py'
urlretrieve(url,SCRIPT)
assert SCRIPT.exists() and SCRIPT.stat().st_size > 5000, 'Training script download failed.'
print('Training script downloaded:',SCRIPT.stat().st_size,'bytes')

In [ ]:
import subprocess, sys
BASE_MODEL='stable-diffusion-v1-5/stable-diffusion-v1-5'
LORA_DIR='/content/noctis-lora'
command=[
    'accelerate','launch','--num_processes=1','--mixed_precision=fp16',str(SCRIPT),
    '--pretrained_model_name_or_path',BASE_MODEL,
    '--train_data_dir',str(TRAIN_DIR),
    '--image_column','image',
    '--caption_column','text',
    '--resolution',str(RESOLUTION),
    '--center_crop',
    '--random_flip',
    '--train_batch_size','1',
    '--gradient_accumulation_steps','1',
    '--gradient_checkpointing',
    '--max_train_steps',str(MAX_TRAIN_STEPS),
    '--learning_rate','1e-4',
    '--lr_scheduler','constant',
    '--lr_warmup_steps','0',
    '--checkpointing_steps','250',
    '--rank','8',
    '--dataloader_num_workers','0',
    '--output_dir',LORA_DIR,
    '--seed',str(SEED),
    '--report_to','tensorboard',
]
print('Starting LoRA training. If the GPU runs out of memory, lower RESOLUTION to 256 and retry.')
subprocess.run(command,check=True)
print('LoRA files:',list(Path(LORA_DIR).glob('*.safetensors')))

## 4. Generate 50 imagined jungle animals

There are 16 species in the site. Each sighting uses the corresponding species cue, a different seed, and an alternate habitat detail. You may revise the prompt list for more creative hybrids. **The trained LoRA adapter is explicitly loaded**, making this a generation step using your own fine-tuned model rather than a generic online image service.

In [ ]:
import gc, torch
from diffusers import StableDiffusionPipeline
from pathlib import Path

species_cues = [
 'moss-covered fox crossed with a forest sloth',
 'luminous moth crossed with a small wild cat',
 'gazelle with transparent crystal antlers',
 'velvet otter crossed with a tiny wild boar',
 'armored hedgehog crossed with a chameleon',
 'long-legged jungle bird crossed with a deer',
 'golden flying squirrel crossed with a butterfly',
 'armadillo crossed with a burrowing salamander',
 'green anole lizard with misty leaf-shaped fins',
 'round forest tapir with glowing amber eyes',
 'young deer with hollow bamboo-like antlers',
 'six-legged iridescent beetle crossed with a bird',
 'anteater with a flower-shaped snout',
 'black feathered owl crossed with a raven',
 'small fox with a giant mushroom-like tail',
 'tree-dwelling lemur with a bioluminescent tail',
]
landscapes=['misty ferns and deep canopy','glowing mushrooms and wet leaves','riverside stones and low fog','moonlit moss and hanging vines','amber twilight and ancient trees']
IMAGES_DIR=Path('/content/images')
IMAGES_DIR.mkdir(exist_ok=True)
pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL,torch_dtype=torch.float16,safety_checker=None)
pipe.load_lora_weights(LORA_DIR,weight_name='pytorch_lora_weights.safetensors')
pipe=pipe.to('cuda')
pipe.enable_attention_slicing()

for idx in range(50):
    cue=species_cues[idx%16]
    place=landscapes[(idx//16+idx)%len(landscapes)]
    prompt=f'portrait of an imaginary wildlife creature, {cue}, {place}, noctisfauna, lush jungle, organic detail, atmospheric light, artistic wildlife photography, beautifully composed'
    generator=torch.Generator(device='cuda').manual_seed(10800+idx)
    picture=pipe(prompt,negative_prompt='words, labels, typography, watermark, low quality, blurry, malformed, duplicate body parts',num_inference_steps=24,guidance_scale=7.0,generator=generator,height=512,width=512).images[0]
    picture.convert('RGB').save(IMAGES_DIR/f'image-{idx+1:03d}.jpg',quality=91,optimize=True)
print('Generated image count:',len(list(IMAGES_DIR.glob('image-*.jpg'))))
assert len(list(IMAGES_DIR.glob('image-*.jpg')))==50

## 5. Review & export images

You should inspect the 50 images, reject poor ones, and regenerate with a new seed or clearer anatomy prompt. **Downloading is optional:** if your browser blocks ZIP downloads, use the next cell's Drive option and copy the exported ZIP from Google Drive instead.

In [ ]:
from PIL import ImageOps, ImageDraw
thumbs=[]
for image_file in sorted(IMAGES_DIR.glob('image-*.jpg')):
    original=Image.open(image_file).convert('RGB')
    thumbs.append(ImageOps.fit(original,(112,112)))
contact=Image.new('RGB',(1120,560),'#0d1a13')
for idx,thumb in enumerate(thumbs): contact.paste(thumb,((idx%10)*112,(idx//10)*112))
display(contact)
contact.save('/content/noctis-contact-sheet.jpg',quality=90)

In [ ]:
from shutil import make_archive, copy2
zip_path=make_archive('/content/images-generated','zip',root_dir='/content',base_dir='images')
print('ZIP:',zip_path)
# OPTION A — Download normally (may be blocked by browser settings):
# from google.colab import files
# files.download(zip_path)
# OPTION B — Save to Drive (recommended when Colab downloads fail):
# from google.colab import drive
# drive.mount('/content/drive')
# copy2(zip_path,'/content/drive/MyDrive/images-generated.zip')
# print('Saved under Google Drive → My Drive → images-generated.zip')

### Finish this part

1. Extract `images-generated.zip` on your computer (or from Google Drive).
2. Copy the **50** `image-001.jpg` … `image-050.jpg` files into `postcards-from-my-jungle/images/`, replacing the demo JPEGs.
3. Do not upload the training source photos or huge training checkpoints to GitHub; only include the generated images and your notebook.
4. Run the sound notebook next; only then change `DEMO_MODE` to `false` in `script.js`.
5. In the README, replace planned-language wording with your actual image-training configuration and observed results.